# 03. Vocabulary Construction & Analysis

This notebook constructs token vocabularies for English and Amharic using the filtered training set (`train_filtered.csv`):
1. Analyzes word frequencies and tests minimum frequency thresholds (`MIN_FREQ`).
2. Evaluates out-of-vocabulary (`<UNK>`) percentages for English and Amharic.
3. Builds deterministic vocabularies with special tokens:
   - `<PAD>` = 0
   - `<UNK>` = 1
   - `<SOS>` = 2
   - `<EOS>` = 3
4. Exports `eng_vocab.json` and `amh_vocab.json` to `paths.vocab_dir`.


In [ ]:
# Environment & Path Setup
# If running on Google Colab, uncomment the lines below:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/english-amharic-nmt

import os
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import get_data_paths
from src.utils.seed import set_seed

# Configure data directory (can be overridden by DATA_ROOT environment variable)
# On Colab: DATA_ROOT = "/content/drive/MyDrive/english-amharic-nmt-data"
DATA_ROOT = os.getenv("DATA_ROOT", str(PROJECT_ROOT / "data"))
paths = get_data_paths(DATA_ROOT)
paths.ensure_directories()
set_seed(42)

print("Project root:", PROJECT_ROOT)
print("Data directory:", paths.data_root)


## 1. Load Filtered Training Data

In [ ]:
import pandas as pd
from collections import Counter

train_df = pd.read_csv(paths.train_filtered_path)
print(f"Training pairs loaded: {len(train_df):,}")


## 2. Token Frequency Analysis
We count token frequencies using whitespace tokenization across the training corpus.

In [ ]:
eng_counter = Counter()
amh_counter = Counter()

for text in train_df["eng"]:
    eng_counter.update(str(text).split())

for text in train_df["amh"]:
    amh_counter.update(str(text).split())

print(f"Unique English tokens: {len(eng_counter):,}")
print(f"Unique Amharic tokens: {len(amh_counter):,}")

print("\nTop 15 most common English tokens:")
print(eng_counter.most_common(15))

print("\nTop 15 most common Amharic tokens:")
print(amh_counter.most_common(15))


## 3. Threshold Experimentation (MIN_FREQ)
We inspect vocabulary sizes across various minimum frequency thresholds.

In [ ]:
thresholds = [1, 2, 3, 5, 10, 20, 50, 100]

print(f"{'Min Frequency':<16} {'English Vocab':>15} {'Amharic Vocab':>15}")
print("-" * 50)
for th in thresholds:
    eng_count = sum(1 for c in eng_counter.values() if c >= th)
    amh_count = sum(1 for c in amh_counter.values() if c >= th)
    print(f"{th:<16} {eng_count:>15,d} {amh_count:>15,d}")


## 4. OOV / UNK Analysis at MIN_FREQ = 5
At `MIN_FREQ = 5`, we calculate the percentage of training tokens mapped to `<UNK>`.

In [ ]:
MIN_FREQ = 5

eng_total = sum(eng_counter.values())
amh_total = sum(amh_counter.values())

eng_unk = sum(c for c in eng_counter.values() if c < MIN_FREQ)
amh_unk = sum(c for c in amh_counter.values() if c < MIN_FREQ)

print(f"English UNK coverage: {eng_unk:,} / {eng_total:,} tokens ({eng_unk/eng_total*100:.4f}%)")
print(f"Amharic UNK coverage: {amh_unk:,} / {amh_total:,} tokens ({amh_unk/amh_total*100:.4f}%)")


## 5. Build and Save Official Vocabularies
Using `build_vocabulary_dict` from `src.data.vocabulary`:
- Special tokens `<PAD>`, `<UNK>`, `<SOS>`, `<EOS>` assigned IDs 0..3.
- All vocabulary tokens with frequency >= 5 are sorted alphabetically.


In [ ]:
from src.data.vocabulary import (
    build_vocabulary_dict,
    save_vocab,
    SPECIAL_TOKENS,
    PAD_IDX, UNK_IDX, SOS_IDX, EOS_IDX
)

eng_vocab = build_vocabulary_dict(train_df["eng"], min_freq=MIN_FREQ)
amh_vocab = build_vocabulary_dict(train_df["amh"], min_freq=MIN_FREQ)

print(f"English vocabulary size: {len(eng_vocab):,}")
print(f"Amharic vocabulary size: {len(amh_vocab):,}")

print("\nSpecial token IDs:")
for tok in SPECIAL_TOKENS:
    print(f"  {tok:<6} -> English: {eng_vocab[tok]} | Amharic: {amh_vocab[tok]}")

# Save vocabularies
save_vocab(eng_vocab, paths.eng_vocab_path)
save_vocab(amh_vocab, paths.amh_vocab_path)
print(f"\nSaved vocabulary files to: {paths.vocab_dir}")
